# Leçon 03 - Modèles de conception agentique

Dans cette leçon, nous explorons trois modèles de conception fondamentaux pour créer des agents IA efficaces :

1. **Instructions claires pour l'agent** — Élaborer des invites précises définissant le rôle qui guident le comportement de l'agent
2. **Sortie structurée avec des modèles Pydantic** — Garantir que les agents renvoient des données prévisibles et validées
3. **Agents à responsabilité unique** — Concevoir des agents ciblés qui excellent chacun dans une tâche spécifique

Nous appliquerons chaque modèle à un scénario de **recommandation de destinations de voyage**, construisant progressivement un système capable de suggérer des destinations, vérifier la disponibilité et gérer la logistique.


## Configuration


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic python-dotenv --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Modèle 1 : Instructions Claires pour l'Agent

Le modèle le plus efficace est aussi le plus simple : rédiger des instructions claires et détaillées pour votre agent.

De bonnes instructions définissent :
- **Qui** est l'agent (persona et ton)
- **Ce qu'**il doit faire (responsabilités étape par étape)
- **Comment** il doit se comporter (contraintes et style)

Ci-dessous, nous créons un agent concierge de voyage avec des instructions explicites qui façonnent chaque réponse qu'il produit.


In [ ]:
agent = provider.as_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Modèle 2 : Sortie structurée avec des modèles Pydantic

Le texte libre est utile pour la conversation, mais les systèmes en aval ont besoin de données structurées.
En associant **des modèles Pydantic** à une **fonction d'outil**, nous pouvons :

- Définir un schéma exact pour la sortie de l'agent
- Valider automatiquement les réponses
- Intégrer de manière fiable les résultats de l'agent dans la logique de l'application

La clé de l'application est de passer `response_format` lorsque nous lançons l'agent. Cela force le
modèle à retourner un objet `TravelRecommendations` validé (disponible dans `response.value`)
au lieu d'un texte libre. L'outil `get_destination_details` retourne également un
`DestinationRecommendation` typé, de sorte que les données restent structurées de bout en bout.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(
    destination: Annotated[str, "The destination to look up"]
) -> DestinationRecommendation:
    """Get structured details about a vacation destination."""
    details = {
        "Barcelona": DestinationRecommendation(
            destination="Barcelona",
            available=True,
            best_season="May-Jun",
            highlights=["Beach", "Architecture", "Nightlife"],
            estimated_budget_usd=2000,
        ),
        "Tokyo": DestinationRecommendation(
            destination="Tokyo",
            available=True,
            best_season="Mar-Apr",
            highlights=["Culture", "Food", "Technology"],
            estimated_budget_usd=2500,
        ),
        "Cape Town": DestinationRecommendation(
            destination="Cape Town",
            available=False,
            best_season="Nov-Mar",
            highlights=["Nature", "Wine", "Adventure"],
            estimated_budget_usd=1800,
        ),
    }
    return details.get(
        destination,
        DestinationRecommendation(
            destination=destination,
            available=False,
            best_season="Unknown",
            highlights=[],
            estimated_budget_usd=0,
        ),
    )


structured_agent = provider.as_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

# Passing `response_format` forces the agent to return a validated
# TravelRecommendations object instead of free-form text.
response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget",
    options={"response_format": TravelRecommendations},
)

if response and response.value:
    result: TravelRecommendations = response.value
    for rec in result.recommendations:
        status = "Available" if rec.available else "Not available"
        print(f"{rec.destination} ({status})")
        print(f"  Best season: {rec.best_season}")
        print(f"  Highlights: {', '.join(rec.highlights)}")
        print(f"  Estimated budget: ${rec.estimated_budget_usd}")
        print()
    print(f"Note: {result.personalized_note}")
else:
    print("No validated structured response was returned.")
    print(response)


## Modèle 3 : Agents à responsabilité unique

Les tâches complexes bénéficient de la répartition du travail entre plusieurs agents spécialisés, chacun ayant une responsabilité unique :

- Un **Expert en destination** qui connaît les lieux et la disponibilité
- Un **Planificateur logistique** qui gère les vols, les hôtels et les itinéraires

Cela reflète le principe d'ingénierie logicielle de *séparation des préoccupations* — chaque agent est plus facile à tester, maintenir et améliorer indépendamment.


In [ ]:
destination_agent = provider.as_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = provider.as_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## Résumé

Dans cette leçon, nous avons appliqué trois modèles de conception agentique à un scénario de recommandation de voyage :

| Modèle | Idée principale | Bénéfice |
|---|---|---|
| **Instructions claires** | Définir le persona, les responsabilités et les contraintes dès le départ | Comportement d'agent cohérent et conforme à la marque |
| **Sortie structurée** | Utiliser des modèles Pydantic comme format de réponse | Résultats validés et lisibles par machine |
| **Responsabilité unique** | Donner à chaque agent une tâche ciblée | Plus facile à tester, maintenir et composer |

Ces modèles se composent naturellement — vous pouvez combiner des instructions claires avec une sortie structurée dans un agent à responsabilité unique pour construire des systèmes robustes et prêts pour la production.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Avertissement** :
Ce document a été traduit à l'aide du service de traduction automatique [Co-op Translator](https://github.com/Azure/co-op-translator). Bien que nous nous efforçions d'assurer l'exactitude, veuillez noter que les traductions automatisées peuvent contenir des erreurs ou des inexactitudes. Le document original dans sa langue native doit être considéré comme la source faisant autorité. Pour les informations critiques, il est recommandé de recourir à une traduction professionnelle réalisée par un humain. Nous ne saurions être tenus responsables des malentendus ou erreurs d'interprétation découlant de l'utilisation de cette traduction.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
